# 02 — Pilot runner
Start with **one task × three conditions**. Do not jump straight to 60 paid runs.

In [1]:
from pathlib import Path
import os, sys, json, yaml, pandas as pd
ROOT=Path.cwd(); ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT
sys.path.insert(0,str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT/'.env')
from src.experiment import load_config, load_instances, run_one, export_predictions
cfg=load_config()
print(cfg['pilot'])

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


This is mini-swe-agent version 2.4.6.
Check the v2 migration guide at https://klieret.short.gy/mini-v2-migration
Loading global config from '/root/.config/mini-swe-agent/.env'

Global cost/call limit: $10.0000 / 80
{'conditions': ['no_memory', 'flat_rag', 'graph'], 'max_tasks': 3, 'dataset_name': 'SWE-bench/SWE-bench_Verified', 'split': 'test', 'environment_class': 'docker', 'exclude_memory_repositories': True}


In [2]:
instances=load_instances(cfg)
[(x['instance_id'],x.get('repo')) for x in instances]

[('django__django-14672', 'django/django'),
 ('sphinx-doc__sphinx-10449', 'sphinx-doc/sphinx'),
 ('django__django-11299', 'django/django')]

### One-task smoke run
This **will call Azure and start a SWE-bench Docker task container**.

In [3]:
task=instances[0]
records=[]
#Uncomment only after notebook 00 + 01 pass.
for condition in cfg['pilot']['conditions']:
    print('RUN',task['instance_id'],condition)
    records.append(run_one(task,condition,cfg))
pd.DataFrame(records)[['instance_id','condition','exit_status','elapsed_seconds','error']]

RUN django__django-14672 no_memory


minisweagent.environment: DEBUG: Starting container with command: docker run -d --name minisweagent-7b684e91 -w    
/testbed --rm docker.io/swebench/sweb.eval.x86_64.django_1776_django-14672:latest sleep 2h

minisweagent.environment: INFO: Started container minisweagent-7b684e91 with ID                                    
1a62c21a7f761943a063df6c460c591e672c2ab29c14a46186e22ef0cf22203c

System:

You are a helpful assistant that can interact with a computer shell to solve programming tasks.

...

Function_call_output:

<exception>action was not executed</exception>
<returncode>-1</returncode>
<output>
</output>

Exit:

diff --git a/django/db/models/fields/reverse_related.py b/django/db/models/fields/reverse_related.py
index ccc2223a5f..65950590e2 100755
--- a/django/db/models/fields/reverse_related.py
+++ b/django/db/models/fields/reverse_related.py
@@ -310,7 +310,7 @@ class ManyToManyRel(ForeignObjectRel):
     def identity(self):
         return super().identity + (
             self.through,
-            self.through_fields,
+            make_hashable(self.through_fields),
             self.db_constraint,
         )

,instance_id,condition,exit_status,elapsed_seconds,error
0,django__django-14672,no_memory,Submitted,50.521473,None
1,django__django-14672,flat_rag,Submitted,57.442080,None
2,django__django-14672,graph,Submitted,37.859169,None


### Export predictions
Official grading must be run separately for each condition/run ID so result caching cannot mix patches.

In [4]:
#Example after records exist:
for condition in cfg['pilot']['conditions']:
    subset=[r for r in records if r['condition']==condition]
    export_predictions(subset, ROOT/'results'/f'predictions_{condition}.jsonl')

### Official SWE-bench evaluation
Use unique `run_id` values for every condition. Start with `max_workers=1`. The evaluation harness applies the patch in Docker and produces the resolved/unresolved result; do not use the agent's self-reported success as the outcome.

In [5]:
from src.evaluate import swebench_eval_command
for condition in cfg['pilot']['conditions']:
    p=ROOT/'results'/f'predictions_{condition}.jsonl'
    print(' '.join(swebench_eval_command(cfg['pilot']['dataset_name'],p,f'pilot_{condition}',max_workers=1)))

python -m swebench.harness.run_evaluation --dataset_name SWE-bench/SWE-bench_Verified --predictions_path /workspace/results/predictions_no_memory.jsonl --max_workers 1 --run_id pilot_no_memory
python -m swebench.harness.run_evaluation --dataset_name SWE-bench/SWE-bench_Verified --predictions_path /workspace/results/predictions_flat_rag.jsonl --max_workers 1 --run_id pilot_flat_rag
python -m swebench.harness.run_evaluation --dataset_name SWE-bench/SWE-bench_Verified --predictions_path /workspace/results/predictions_graph.jsonl --max_workers 1 --run_id pilot_graph


After one clean task, raise `pilot.max_tasks` to 3 for a smoke pilot. Only after that should you predeclare ~20 held-out tasks and run all three conditions.